In [1]:
# epoch影响较大
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project/database/Dataset42/Dataset42_P9_SC.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project/database/Dataset42/Dataset42_ST.h5ad" \
    --n_epochs 300 \
    --resolution 2 \
    --loss_type zinb \
    --beta 0.1 \
    --lambda_mmd 1.0 \
    --top_n_per_type 100 \
    --latent_dim 256 \
    --output_dir ../deconv_results/Dataset42_P9 \
    --aggregation_method mean
 #   --precomputed_marker_file "/mnt/d/ST_Graduation_Project/SC_MAP_ST/deconv_results/GSE243275/final_genes.txt"

/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 2.0
   Batch size: 256
   Epochs: 300
   Learning rate: 0.001
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: ZINB
   Lambda MMD: 1.0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project/database/Dataset42/Dataset42_P9_SC.h5ad
   SC shape: (6519, 24542)
   SC avg counts/cell: 12108.8 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project/database/Dataset42/Dataset42_ST.h5ad
   ST shape: (1933, 17823)
   Common genes: 16078
   SC final: (6519, 16078)
   ST final: (1933, 16078)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 43 clusters
Marker genes per cluster:
   0: 100 -> 100 (after Lasso)
   1: 100 -> 100 (after Lasso)
   10: 100 -> 100 (after Lasso)
   11: 100 -> 100 (after Lasso)
   12: 100 -> 100 (after Lasso)
   13

VAE Training:  50%|████▉     | 149/300 [03:12<03:15,  1.29s/epoch, Train=1454.4766, Recon=1448.6739, KL=58.0269, MMD=0.0114, Test=1396.6943]



⚠️ Early stopping triggered at epoch 150/300
   Best test loss: 1391.9434, Current test loss: 1396.6943
   No improvement for 20 consecutive epochs
📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (6519, 1314)
   Number of clusters: 43
   Computed embeddings shape: (6519, 256)
Computing cluster centers with mean aggregation...
      Cluster 0: 449 cells (mean aggregation)
      Cluster 1: 416 cells (mean aggregation)
      Cluster 2: 225 cells (mean aggregation)
      Cluster 3: 195 cells (mean aggregation)
      Cluster 4: 186 cells (mean aggregation)
      Cluster 5: 177 cells (mean aggregation)
      Cluster 6: 171 cells (mean aggregation)
      Cluster 7: 162 cells (mean aggregation)
      Cluster 8: 161 cells (mean aggregation)
      Cluster 9: 150 cells (mean aggregation)
      Cluster 10: 126 cells (mean aggregation)
      Cluster 11: 125 cells (mean aggregation)
      Cluster 12: 352 cells (mean aggregation)
      Cluster 13: 121 cells (mean aggr

In [4]:
# λ 越大更稀疏0.1-0.5
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project/database/Dataset42/Dataset42_ST.h5ad" \
    --stage1_model_path "../deconv_results/Dataset42_P9/final_vae.pth" \
    --output_dir "../deconv_results/Dataset42_P9" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 1e-4 \
    --loss_lambda_mse 0.1 \
    --loss_lambda_cosine 5 \
    --loss_lambda_pearson 5 \
    --loss_lambda_reg 0.1 \
    --loss_lambda_sparse 0.1 \
    --loss_lambda_proportion 0.1 \
    --k_spatial 10 \
    --k_celltype 20 \
    --batch_size 512 \
    --n_epochs 400 \
    --weight_threshold 0.05

Sample name: Dataset42
Stage 1 model: ../deconv_results/Dataset42_P9/final_vae.pth
Output directory: ../deconv_results/Dataset42_P9
Device: cpu
Weight threshold: 0.05
Loading pretrained VAE Encoder...
   VAE architecture: 1314 -> 256
   Output type: zinb
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 6519 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/Dataset42_P9/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([43, 256])
   Cluster expressions (marker): torch.Size([43, 1314])
   Cluster expressions (all genes): 43 × 16078
   Loaded celltype mapping: 43 clusters
   Average cell counts: 12108.8
Loaded all genes list: 16078 genes
VAE Encoder loaded: 1314 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '5', '6', '7', '8', '9']
Mark

GAT Training:  11%|█▏        | 45/400 [03:03<24:06,  4.08s/epoch, Total=108.9538, Pearson=0.5715, MSE=1035.9116, Cosine=0.4734, P_pat=10, M_pat=10, C_pat=10]



⚠️ Early stopping triggered at epoch 46/400
   All three core losses stopped improving:
      Pearson: best=0.5709, current=0.5715, no improvement for 10 epochs
      MSE: best=1035.8045, current=1035.9116, no improvement for 10 epochs
      Cosine: best=0.4729, current=0.4734, no improvement for 10 epochs
Evaluating model results...
Applying weight threshold: 0.05
   Non-zero elements: 38660 -> 4991 (6.0%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/Dataset42_P9/Dataset42_reconstructed_all_genes.csv
   Cell type composition...
   Found duplicate celltype names: ['CD1C', 'Endothelial Cell', 'Epithelial', 'Mac', 'Fibroblast', 'Tcell', 'CLEC9A']. Merging corresponding cluster columns by summing weights.
   Columns before: 43, after merge: 13
   Saved cell composition (celltype): ../deconv_results/Dataset42_P9/Dataset42_cell_composition.csv
   Saved cluster composition: ../deconv_resu